In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import pickle
import seaborn as sns
sns.set(style="whitegrid")


import platform
system_name = platform.system()
if system_name == 'Windows':
    plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 用黑体
elif system_name == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac 用 Arial Unicode

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题



In [3]:
###########################################################################################

ann = pd.read_csv(r"E:\26Spring\Final\data\hk_announcement.csv", encoding="gbk")
em = pd.read_csv(r"E:\26Spring\Final\code\em_buyback_filtered.csv")

# bs = pd.read_csv("E:\\25Winter\\4080\data2\hk_data\HK_balancesheet.csv")
# cf = pd.read_csv("E:\\25Winter\\4080\data3\hk_cashflow_statement.csv")
# inc = pd.read_csv(r"E:\25Winter\4080\data3\hk_income_statement.csv",engine="python",on_bad_lines="skip")

hsci = pd.read_csv(r"E:\26Spring\Final\data\HSCI.csv")
mktcap = pd.read_csv(r"E:\26Spring\Final\data\hk_mktcap_long.csv")
dy = pd.read_hdf(r"E:\26Spring\Final\data\hk_dividendyield.h5")
price = pd.read_csv(r"E:\26Spring\Final\data\hk_price.csv")
share = pd.read_hdf(r"E:\26Spring\Final\data\hk_shares.h5")


In [4]:
###########################################################################################

def normalize_ticker(x):
    """
    把股票代码统一成 4 位 + .hk 的格式
    例如:
    5      -> 0005.hk
    700    -> 0700.hk
    0005.HK -> 0005.hk
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper().replace(".HK", "")
    x = x.zfill(4)
    return x + ".hk"

def preprocess_df(df, ticker_col, date_col):
    """
    通用预处理:
    1. 复制数据
    2. 日期列转 datetime
    3. 筛选 2015-01-01 之后
    4. ticker 标准化
    5. 列名统一成 ticker / date
    """
    df = df.copy()

    # 先重命名，后面统一处理
    df = df.rename(columns={
        ticker_col: "ticker",
        date_col: "date"
    })

    # 日期格式
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # ticker 格式
    df["ticker"] = df["ticker"].apply(normalize_ticker)

    # 筛选 2015 年之后
    df = df[df["date"] >= pd.Timestamp("2015-01-01")].copy()

    return df

def preprocess_df(df, ticker_col, date_col):
    """
    通用预处理:
    1. 复制数据
    2. 日期列转 datetime
    3. 筛选 2012-01-01 之后
    4. ticker 标准化
    5. 列名统一成 ticker / date
    """
    df = df.copy()

    # 先重命名，后面统一处理
    df = df.rename(columns={
        ticker_col: "ticker",
        date_col: "date"
    })

    # 日期格式
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # ticker 格式
    df["ticker"] = df["ticker"].apply(normalize_ticker)

    # 筛选 2015 年之后
    df = df[df["date"] >= pd.Timestamp("2015-01-01")].copy()

    return df

def filter_hsci(df, hsci, hsci_ticker_col="ticker", hsci_date_col="date"):
    """
    用每日动态 HSCI 成分股筛选
    要求:
    - df 中已经有 date / ticker
    - hsci 原始列名可以传入
    """
    df = df.copy()
    hsci = hsci.copy()

    # 统一 hsci 列名
    hsci = hsci.rename(columns={
        hsci_ticker_col: "ticker",
        hsci_date_col: "date"
    })

    # 标准化 hsci
    hsci["date"] = pd.to_datetime(hsci["date"], errors="coerce")
    hsci["ticker"] = hsci["ticker"].apply(normalize_ticker)

    # 只保留有效键
    hsci = hsci[["date", "ticker"]].dropna().drop_duplicates()

    # inner merge 做筛选，比 set+list comprehension 更稳
    out = df.merge(
        hsci,
        on=["date", "ticker"],
        how="inner"
    )

    return out



In [5]:
price_clean = preprocess_df(
    price,
    ticker_col="sid",
    date_col="date"
)

price_hsci = filter_hsci(
    price_clean,
    hsci,
    hsci_ticker_col="sid",  
    hsci_date_col="date"
)

price_hsci = price_hsci[
    ["date", "ticker", "AdjClose", "close", "open", "high", "low", "volume", "amount", "turnover"]
].copy()

#######################################

dy_df = dy.rename("dy").reset_index()

dy_clean = preprocess_df(
    dy_df,
    ticker_col="sid",
    date_col="date"
)

dy_hsci = filter_hsci(
    dy_clean,
    hsci,
    hsci_ticker_col="sid",
    hsci_date_col="date"
)

dy_hsci = dy_hsci[
    ["date", "ticker", "dy"]
].copy()

#######################################

mktcap2 = mktcap.copy()
mktcap2 = mktcap2.rename(columns={"mkt_cap_ard": "mktcp"})

mktcap_clean = preprocess_df(
    mktcap2,
    ticker_col="stock",
    date_col="date"
)

mktcap_hsci = filter_hsci(
    mktcap_clean,
    hsci,
    hsci_ticker_col="sid",
    hsci_date_col="date"
)

mktcap_hsci = mktcap_hsci[
    ["date", "ticker", "mktcp"]
].copy()

#######################################

em2 = em.copy()
em2["buyback_amount"] = em2["回购总额"]

em_clean = preprocess_df(
    em2,
    ticker_col="股票代码",
    date_col="日期"
)

em_hsci = filter_hsci(
    em_clean,
    hsci,
    hsci_ticker_col="sid",
    hsci_date_col="date"
)

em_hsci = em_hsci[
    ["date", "ticker", "buyback_amount"]
].copy()

#######################################


In [6]:
panel = price_hsci.merge(
    dy_hsci,
    on=["date", "ticker"],
    how="left"
)

panel = panel.merge(
    em_hsci,
    on=["date", "ticker"],
    how="left"
)

panel = panel.merge(
    mktcap_hsci,
    on=["date", "ticker"],
    how="left"
)

In [7]:
panel

,date,ticker,AdjClose,close,open,high,low,volume,amount,turnover,dy,buyback_amount,mktcp
0,2015-01-02,0001.hk,267.426000,131.70,131.00,132.30,130.20,2331309.0,3.067293e+05,NaN,NaN,NaN,NaN
1,2015-01-02,0002.hk,299.715800,67.25,67.00,67.45,66.40,1649366.0,1.106447e+05,NaN,NaN,NaN,NaN
2,2015-01-02,0003.hk,738.075600,17.76,17.78,17.86,17.38,4817803.0,8.518497e+04,NaN,NaN,NaN,NaN
3,2015-01-02,0004.hk,147.770000,57.25,56.50,57.50,56.10,3757908.0,2.143517e+05,NaN,NaN,NaN,NaN
4,2015-01-02,0005.hk,609.385700,74.00,73.60,74.10,73.50,8614012.0,6.353929e+05,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301860,2025-09-30,9988.hk,186.405068,177.00,176.60,178.00,173.70,135460219.0,2.389275e+10,0.840444,1.109045,NaN,3.375856e+12
1301861,2025-09-30,9992.hk,273.973118,266.80,260.20,268.00,260.00,11330044.0,2.995829e+09,1.645465,0.332871,NaN,3.582972e+11
1301862,2025-09-30,9993.hk,2.986319,2.67,2.60,2.68,2.60,4506000.0,1.191838e+07,0.698359,NaN,NaN,1.080076e+10
1301863,2025-09-30,9997.hk,12.055474,8.89,8.86,8.89,8.86,980500.0,8.706575e+06,0.278707,2.939258,NaN,1.073907e+10


In [11]:
panel.to_csv(r"E:\26Spring\Final\code\panel.csv", index=False)